In [1]:
import pandas as pd

# Loading the data
df = pd.read_csv(
    "C:/Users/adaml/OneDrive/Documents/Data Projects/Food_items_for_meal_engine.csv",
    encoding="cp1252",
)
df.columns = df.columns.str.strip()

print("=== CODESPACE POOL DIAGNOSTICS ===")
print(f"Total rows found in CSV: {len(df)}")

# =====================================================================
# CHECKING MEAL STRUCTURE INTEGRITY
# =====================================================================

# 1. Testing Meal Types
b_rows = df[df["Meal_type"].str.contains("Breakfast", case=False, na=False)]
ld_rows = df[df["Meal_type"].str.contains("Lunch|Dinner", case=False, na=False)]
print(f"- Rows matching 'Breakfast': {len(b_rows)}")
print(f"- Rows matching 'Lunch|Dinner': {len(ld_rows)}")

# 2. Test Category Types across your specific combinations
print("\n=== BREAKDOWN OF CATEGORY TYPES IN YOUR CSV ===")
print(df["Category_type"].value_counts())

print("\n=== ENGINE MEAL STRUCTURE INTEGRITY CHECK ===")
pools = {
    "Breakfast Savory": b_rows[b_rows["Category_type"] == "Savory"],
    "Breakfast Starch": b_rows[b_rows["Category_type"] == "Starch"],
    "Breakfast Sweet/Fruit": b_rows[b_rows["Category_type"].isin(["Fruit", "Dessert", "Supplement"])],
    "Lunch/Dinner Savory": ld_rows[ld_rows["Category_type"] == "Savory"],
    "Lunch/Dinner Starch": ld_rows[ld_rows["Category_type"] == "Starch"],
    "Lunch/Dinner Vegetable": ld_rows[ld_rows["Category_type"] == "Vegetable"],
}

errors_found = False
for pool_name, pool_df in pools.items():
    if pool_df.empty:
        print(f"❌ CRITICAL ERROR: '{pool_name}' pool is EMPTY! Your code is deadlocking here.")
        errors_found = True
    else:
        print(f"✅ SUCCESS: '{pool_name}' pool has {len(pool_df)} available items.")

if not errors_found:
    print("\nDataFrame integrity looks healthy. Let's inspect macro numbers...")
    print(df[["Calories_per_100g", "Protein_g", "Fat_g", "Carbs_g", "Serving_size_g"]].dtypes)

=== CODESPACE POOL DIAGNOSTICS ===
Total rows found in CSV: 21
- Rows matching 'Breakfast': 6
- Rows matching 'Lunch|Dinner': 16

=== BREAKDOWN OF CATEGORY TYPES IN YOUR CSV ===
Category_type
Savory        11
Starch         3
Vegetable      3
Supplement     1
Dessert        1
Snack          1
Fruit          1
Name: count, dtype: int64

=== ENGINE MEAL STRUCTURE INTEGRITY CHECK ==
✅ SUCCESS: 'Breakfast Savory' pool has 1 available items.
✅ SUCCESS: 'Breakfast Starch' pool has 1 available items.
✅ SUCCESS: 'Breakfast Sweet/Fruit' pool has 3 available items.
✅ SUCCESS: 'Lunch/Dinner Savory' pool has 10 available items.
✅ SUCCESS: 'Lunch/Dinner Starch' pool has 2 available items.
✅ SUCCESS: 'Lunch/Dinner Vegetable' pool has 3 available items.

DataFrame integrity looks healthy. Let's inspect macro numbers...
Calories_per_100g      int64
Protein_g            float64
Fat_g                float64
Carbs_g              float64
Serving_size_g         int64
dtype: object


In [2]:
import numpy as np
import pandas as pd

# =====================================================================
# 1. LOADING & CLEANING THE DATA
# =====================================================================
df = pd.read_csv(
    "C:/Users/adaml/OneDrive/Documents/Data Projects/Food_items_for_meal_engine.csv",
    encoding="cp1252",
)

df.columns = df.columns.str.strip()

df["Cost per 100g(£)"] = (
    df["Cost per 100g(£)"]
    .astype(str)
    .str.replace("£", "")
    .str.strip()
)
df["Cost per 100g(£)"] = pd.to_numeric(df["Cost per 100g(£)"], errors="coerce").fillna(0.0)


# =====================================================================
# 2. DEFINING MACRO TARGETS BY DAY TYPE
# =====================================================================
def get_daily_targets(day_type):
    target_calories = 2050.0
    if day_type == "Gym Day":
        return {
            "calories": target_calories,
            "protein": 154.0,
            "carbs": 256.0,
            "fats": 46.0,
        }
    elif day_type == "Running Day":
        return {
            "calories": target_calories,
            "protein": 103.0,
            "carbs": 282.0,
            "fats": 57.0,
        }


# =====================================================================
# 3. HELPER FUNCTION: QUANTITY AND MACRO METRIC SCALAR
# =====================================================================
def calculate_item_macros(row, quantity=1.0):
    scalar = (row["Serving_size_g"] / 100.0) * quantity
    return {
        "name": row["Name"],
        "unit_type": row["Unit_type"],
        "serving_size_g": row["Serving_size_g"],
        "quantity": quantity,
        "calories": row["Calories_per_100g"] * scalar,
        "protein": row["Protein_g"] * scalar,
        "fats": row["Fat_g"] * scalar,
        "carbs": row["Carbs_g"] * scalar,
        "cost": row["Cost per 100g(£)"] * scalar,
        "prep_time": row["Prep_time(mins)"],
        "raw_row": row,
    }


# =====================================================================
# 4. MAIN MENU GENERATION ENGINE (WITH TOLERANT FALLBACK LOGIC)
# =====================================================================
def generate_daily_menu(df, day_type, max_available_prep_time=160):
    targets = get_daily_targets(day_type)
    p_tol, c_tol, f_tol = 0.05, 0.05, 0.05

    attempts = 0
    while attempts < 5000:
        attempts += 1
        daily_menu = []

        current_p_tol = p_tol + (attempts // 100 * 0.002)
        current_c_tol = c_tol + (attempts // 100 * 0.002)
        current_f_tol = f_tol + (attempts // 100 * 0.002)

        if max_available_prep_time <= 5:
            snack_pool = df[df["Category_type"].isin(["Snack", "Supplement"])]
            if snack_pool.empty:
                continue
            sampled_items = snack_pool.sample(n=2, replace=True)
            for _, item in sampled_items.iterrows():
                daily_menu.append(calculate_item_macros(item, quantity=2.0))
        else:
            # --- 1. EXTRACTING STRUCTURAL MEAL SUB-POOLS ---
            b_pool = df[df["Meal_type"].str.contains("Breakfast", case=False, na=False)]
            b_savory = b_pool[b_pool["Category_type"] == "Savory"]
            b_starch = b_pool[b_pool["Category_type"] == "Starch"]
            b_sweet = b_pool[b_pool["Category_type"].isin(["Fruit", "Dessert", "Supplement"])]

            ld_pool = df[df["Meal_type"].str.contains("Lunch|Dinner", case=False, na=False)]
            ld_savory = ld_pool[ld_pool["Category_type"] == "Savory"]
            ld_starch = ld_pool[ld_pool["Category_type"] == "Starch"]
            ld_veg = ld_pool[ld_pool["Category_type"] == "Vegetable"]

            # GLOBAL CRITICAL FALLBACKS: If structural groups are missing entirely from the CSV
            all_starches = df[df["Category_type"] == "Starch"]
            all_savories = df[df["Category_type"] == "Savory"]
            all_sweets = df[df["Category_type"].isin(["Fruit", "Dessert", "Supplement", "Snack"])]

            # CORE FAILURE CHECK: If the database is missing starches or proteins entirely, abort
            if all_starches.empty or all_savories.empty or ld_veg.empty:
                print("CRITICAL: The database is completely missing core Starch, Savory, or Vegetable rows.")
                return None, 0, 0, 0, 0, 0

            # --- 2. DYNAMIC FALLBACK SELECTION SEEDING ---
            # Breakfast Savory Fallback
            b_sav_item = b_savory.sample().iloc[0] if not b_savory.empty else all_savories.sample().iloc[0]
            
            # Breakfast Starch Fallback (If no Breakfast Starch, pull safely from Lunch/Dinner Starches)
            b_sta_item = b_starch.sample().iloc[0] if not b_starch.empty else all_starches.sample().iloc[0]
            
            # Breakfast Sweet Fallback
            b_sw_item = b_sweet.sample().iloc[0] if not b_sweet.empty else all_sweets.sample().iloc[0]

            # Lunch / Dinner Fallbacks
            l_sav_item = ld_savory.sample().iloc[0] if not ld_savory.empty else all_savories.sample().iloc[0]
            l_sta_item = ld_starch.sample().iloc[0] if not ld_starch.empty else all_starches.sample().iloc[0]
            l_veg_item = ld_veg.sample().iloc[0]

            d_sav_item = ld_savory.sample().iloc[0] if not ld_savory.empty else all_savories.sample().iloc[0]
            d_sta_item = ld_starch.sample().iloc[0] if not ld_starch.empty else all_starches.sample().iloc[0]
            d_veg_item = ld_veg.sample().iloc[0]

            # Append the safely managed configurations to the blueprint
            daily_menu.append(calculate_item_macros(b_sav_item))
            daily_menu.append(calculate_item_macros(b_sta_item))
            daily_menu.append(calculate_item_macros(b_sw_item))

            daily_menu.append(calculate_item_macros(l_sav_item))
            daily_menu.append(calculate_item_macros(l_sta_item))
            daily_menu.append(calculate_item_macros(l_veg_item))

            daily_menu.append(calculate_item_macros(d_sav_item))
            daily_menu.append(calculate_item_macros(d_sta_item))
            daily_menu.append(calculate_item_macros(d_veg_item))

        # =====================================================================
        # GLOBAL PORTION FAT COMPRESSION (in the event fats get too high)
        # =====================================================================
        initial_fats = sum(item["fats"] for item in daily_menu)
        fat_safety_ceiling = targets["fats"] * 0.90

        if initial_fats > fat_safety_ceiling and initial_fats > 0:
            universal_downscaler = fat_safety_ceiling / initial_fats
            for item in daily_menu:
                item["quantity"] *= universal_downscaler

        # --- DYNAMIC TARGET SCALE ADJUSTMENTS ---
        # 1. Protein Scaling
        temp_menu = [calculate_item_macros(item["raw_row"], quantity=item["quantity"]) for item in daily_menu]
        base_p = sum(item["protein"] for item in temp_menu)
        p_gap = targets["protein"] - base_p

        if p_gap > 0 and base_p > 0:
            clean_protein_items = [i for i in daily_menu if i["protein"] > (i["fats"] * 1.2) and i["protein"] > 2]
            if clean_protein_items:
                p_pool_total = sum(i["protein"] for i in clean_protein_items)
                if p_pool_total > 0:
                    p_multiplier = 1.0 + (p_gap / p_pool_total)
                    for item in clean_protein_items:
                        item["quantity"] *= min(p_multiplier, 3.5)

        # 2. Carbohydrate Scaling
        temp_menu = [calculate_item_macros(item["raw_row"], quantity=item["quantity"]) for item in daily_menu]
        base_c = sum(item["carbs"] for item in temp_menu)
        c_gap = targets["carbs"] - base_c

        if c_gap > 0 and base_c > 0:
            clean_carb_items = [i for i in daily_menu if i["carbs"] > (i["fats"] * 2.5) and i["carbs"] > 8]
            if clean_carb_items:
                c_pool_total = sum(i["carbs"] for i in clean_carb_items)
                if c_pool_total > 0:
                    c_multiplier = 1.0 + (c_gap / c_pool_total)
                    for item in clean_carb_items:
                        item["quantity"] *= min(c_multiplier, 4.0)

        # 3. Optional Snack Supplement Filler
        temp_menu = [calculate_item_macros(item["raw_row"], quantity=item["quantity"]) for item in daily_menu]
        final_p = sum(item["protein"] for item in temp_menu)
        final_c = sum(item["carbs"] for item in temp_menu)
        if final_p < targets["protein"] or final_c < targets["carbs"]:
            snack_pool = df[df["Category_type"].isin(["Snack", "Supplement"])]
            if not snack_pool.empty:
                daily_menu.append(calculate_item_macros(snack_pool.sample().iloc[0], quantity=1.0))

        # --- RECOMPUTING ACTUAL FINALS FROM RAW DATA ---
        finalized_menu = [calculate_item_macros(item["raw_row"], quantity=item["quantity"]) for item in daily_menu]

        total_p = sum(item["protein"] for item in finalized_menu)
        total_c = sum(item["carbs"] for item in finalized_menu)
        total_f = sum(item["fats"] for item in finalized_menu)
        total_cost = sum(item["cost"] for item in finalized_menu)
        total_cal = (total_p * 4) + (total_c * 4) + (total_f * 9)

        if total_cost > 11.42:
            continue

        p_pass = targets["protein"] * (1 - current_p_tol) <= total_p <= targets["protein"] * (1 + current_p_tol)
        c_pass = targets["carbs"] * (1 - current_c_tol) <= total_c <= targets["carbs"] * (1 + current_c_tol)
        f_pass = targets["fats"] * (1 - current_f_tol) <= total_f <= targets["fats"] * (1 + current_f_tol)

        if p_pass and c_pass and f_pass:
            print(f"Success! Optimal menu variant resolved on attempt {attempts} (Tolerance: {current_c_tol*100:.1f}%).")
            return finalized_menu, total_cal, total_p, total_c, total_f, total_cost

    return None, 0, 0, 0, 0, 0


# =====================================================================
# 5. EXECUTING MEAL GENERATOR AND PRINTING SEGMENTED MENU
# =====================================================================
menu, calories, p, c, f, daily_spend = generate_daily_menu(
    df, day_type="Gym Day", max_available_prep_time=120
)

if menu:
    print("\n" + "="*60)
    print(f" DAILY TOTALS: {calories:.1f} kcal | P: {p:.1f}g | C: {c:.1f}g | F: {f:.1f}g | Spend: £{daily_spend:.2f}")
    print("="*60)
    
    print("\n## BREAKFAST")
    for item in menu[0:3]:
        print(f"  - {item['quantity']:.2f} x {item['name']} ({item['serving_size_g'] * item['quantity']:.0f}g total)")
        
    print("\n## LUNCH")
    for item in menu[3:6]:
        print(f"  - {item['quantity']:.2f} x {item['name']} ({item['serving_size_g'] * item['quantity']:.0f}g total)")
        
    print("\n## DINNER")
    for item in menu[6:9]:
        print(f"  - {item['quantity']:.2f} x {item['name']} ({item['serving_size_g'] * item['quantity']:.0f}g total)")
        
    if len(menu) > 9:
        print("\n## SNACKS & SUPPLEMENTS")
        for item in menu[9:]:
            print(f"  - {item['quantity']:.2f} x {item['name']} ({item['serving_size_g'] * item['quantity']:.0f}g total)")
    print("\n" + "="*60)
            
else:
    print("Could not discover an allocation variant matching constraints. Loosen boundaries.")

Success! Optimal menu variant resolved on attempt 3 (Tolerance: 5.0%).

 DAILY TOTALS: 2103.1 kcal | P: 157.5g | C: 267.6g | F: 44.7g | Spend: £8.24

## BREAKFAST
  - 2.22 x Egg Whites (222g total)
  - 2.52 x NYB Bagels (plain) (214g total)
  - 1.14 x Banana (134g total)

## LUNCH
  - 2.22 x Chicken Drumstick(S) (84g total)
  - 1.00 x Indomie Mi Goreng (80g total)
  - 1.00 x Cucumber (23g total)

## DINNER
  - 2.22 x Chicken Breast (222g total)
  - 2.52 x White Rice (252g total)
  - 2.22 x Broccoli (222g total)

